``SeqMut`` measures the model-free change a mutation induces in a set of CPP features (ΔCPP). We first build a feature set with a small ``CPP`` run, then :meth:`SeqMut.scan` enumerates every TMD substitution and ranks them by ``delta_cpp`` (the L1 magnitude of the feature change).

In [1]:
import aaanalysis as aa
aa.options["verbose"] = False

df_seq = aa.load_dataset(name="DOM_GSEC", n=10)
labels = df_seq["label"].to_list()
sf = aa.SequenceFeature()
df_parts = sf.get_df_parts(df_seq=df_seq)
split_kws = sf.get_split_kws()
cpp = aa.CPP(df_parts=df_parts, split_kws=split_kws, verbose=False)
df_feat = cpp.run(labels=labels, n_filter=25)

seqm = aa.SeqMut()
df_scan = seqm.scan(df_seq=df_seq, df_feat=df_feat, region="tmd")
aa.display_df(df_scan, n_rows=10, show_shape=True)

DataFrame shape: (8740, 8)


,entry,pos,from_aa,to_aa,mutation,region,delta_cpp,shift_score
1,P16070,669,A,P,A669P,tmd,4.046420,-3.934420
2,P16070,665,A,P,A665P,tmd,3.975670,-3.863670
3,P09803,730,L,P,L730P,tmd,3.523250,-3.355250
4,Q03157,604,L,P,L604P,tmd,3.523250,-3.355250
5,P05556,748,L,P,L748P,tmd,3.523250,-3.355250
6,Q06481,713,L,P,L713P,tmd,3.523250,-3.355250
7,P05067,720,L,P,L720P,tmd,3.523250,-3.355250
8,P16070,669,A,G,A669G,tmd,3.422670,-3.422670
9,P70180,492,L,P,L492P,tmd,3.417250,-3.249250
10,P01135,114,L,P,L114P,tmd,3.417250,-3.249250


``region`` can be a part name (``'jmd_n'`` / ``'tmd'`` / ``'jmd_c'``), ``None`` (the full JMD-N + TMD + JMD-C span), or a list of 1-based positions; ``to_aa`` restricts the substitution alphabet.

In [2]:
# to_aa restricts the substitution alphabet (here: a hydrophobic + proline set);
# jmd_n_len / jmd_c_len set the JMD lengths used to classify each position's region
df_scan_sub = seqm.scan(df_seq=df_seq, df_feat=df_feat,
                          region="tmd", to_aa=["A", "L", "V", "P"],
                          jmd_n_len=10, jmd_c_len=10)
aa.display_df(df_scan_sub, n_rows=10, show_shape=True)

DataFrame shape: (1593, 8)


,entry,pos,from_aa,to_aa,mutation,region,delta_cpp,shift_score
1,P16070,669,A,P,A669P,tmd,4.046420,-3.934420
2,P16070,665,A,P,A665P,tmd,3.975670,-3.863670
3,Q03157,604,L,P,L604P,tmd,3.523250,-3.355250
4,P09803,730,L,P,L730P,tmd,3.523250,-3.355250
5,Q06481,713,L,P,L713P,tmd,3.523250,-3.355250
6,P05556,748,L,P,L748P,tmd,3.523250,-3.355250
7,P05067,720,L,P,L720P,tmd,3.523250,-3.355250
8,P16234,542,L,P,L542P,tmd,3.417250,-3.249250
9,P70180,492,L,P,L492P,tmd,3.417250,-3.249250
10,P01135,114,L,P,L114P,tmd,3.417250,-3.249250


**Design constraints.** `constraints` takes a shared `DesignConstraints` object that replaces the `region` / `to_aa` shorthands and additionally excludes immutable positions and forbidden target residues from the scan:

In [3]:
tmd_start = int(df_seq["tmd_start"].iloc[0])
dc = aa.DesignConstraints(mutable_positions="tmd",
                          permitted_substitutions=["A", "L", "V", "P"],
                          immutable_positions=[tmd_start],   # keep the first TMD residue
                          forbidden_substitutions=["P"])     # never introduce a proline
df_scan_dc = seqm.scan(df_seq=df_seq, df_feat=df_feat, constraints=dc)
aa.display_df(df_scan_dc, n_rows=10, show_shape=True)


DataFrame shape: (1135, 8)


,entry,pos,from_aa,to_aa,mutation,region,delta_cpp,shift_score
1,P05556,744,G,A,G744A,tmd,3.415670,3.415670
2,Q8IUW5,74,G,A,G74A,tmd,3.415670,3.415670
3,P53801,112,G,A,G112A,tmd,3.415660,3.415660
4,Q14802,52,G,A,G52A,tmd,3.415660,3.415660
5,Q8IUW5,78,C,A,C78A,tmd,2.859590,2.859590
6,P53801,116,C,A,C116A,tmd,2.859580,2.859580
7,P01135,118,C,A,C118A,tmd,2.859580,2.859580
8,Q969W9,60,C,A,C60A,tmd,2.859580,2.859580
9,P05556,744,G,L,G744L,tmd,2.801250,2.801250
10,Q14802,52,G,L,G52L,tmd,2.801250,2.801250
